# Multi-Molecule GFlowNet On Colab

This notebook is the Colab counterpart for GFlowNet. It bootstraps the `gflownet_v2` branch in `/content/Thesis`, uses `scripts/init_colab.py` only for dependency installation and mini post-training dataset preparation, then runs `scripts/train_multi_molecule_gflownet.py` directly with a temporary runtime config. It supports `tb` and `db`, defaults to conservative masked rollout sampling, prefers the mini multi-molecule SFT checkpoint from `colab/02_train_multi_molecule_sft.ipynb`, and falls back to the original BioT5 ChEBI-20 weights when that checkpoint is absent.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/mruniverse8/Thesis.git"
REPO_BRANCH = "gflownet_v2"
REPO_DIR = Path("/content/Thesis")

%cd /content
if (REPO_DIR / ".git").exists():
    print(f"Reusing {REPO_DIR}")
elif REPO_DIR.exists():
    raise RuntimeError(f"Existing non-git directory at {REPO_DIR}; delete it and rerun the notebook.")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", REPO_URL, REPO_BRANCH], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-B", REPO_BRANCH, "FETCH_HEAD"], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print({"repo_dir": str(REPO_DIR), "repo_branch": REPO_BRANCH})


In [ ]:
DATASET_MODE = "auto"
TRAIN_DATASET_FILE_ID = "1fwRIHrcq0nGA1OJCCWqdxvGgcbW2oKY3"
CONFIG_OVERRIDE = None
GFLOWNET_OBJECTIVE = "tb"  # switch to "db" to test Detailed Balance
DEFAULT_CONFIG_STEM = "multi_molecule_gflownet_mini"
FORCE_VALID_MASKING = True
SELFIES_DICT_PATH = "molecules/dict/selfies_dict.txt"
ROLLOUT_TEMPERATURE = 0.08
ROLLOUT_TOP_P = 0.25

BASE_TRAIN_CONFIG = CONFIG_OVERRIDE or Path("configs") / f"{DEFAULT_CONFIG_STEM}.yaml"
WANDB_PROJECT_URL = "https://wandb.ai/koala-team/Thesis-2"

OUTPUT_DIR = REPO_DIR / "outputs" / f"{BASE_TRAIN_CONFIG.stem}_{GFLOWNET_OBJECTIVE}"
OUTPUT_DIR_ZIP = OUTPUT_DIR.with_suffix(".zip")
CHECKPOINTS_DIR = OUTPUT_DIR / "checkpoints"
BEST_CHECKPOINT_DIR = CHECKPOINTS_DIR / "best"
BEST_CHECKPOINT_ZIP = CHECKPOINTS_DIR / "best.zip"
RUN_SUMMARY_PATH = OUTPUT_DIR / "run_summary.json"
ITERATION_DIAGNOSTICS_PATH = OUTPUT_DIR / "diagnostics" / "iteration_diagnostics.jsonl"
TRAJECTORY_PREVIEWS_PATH = OUTPUT_DIR / "diagnostics" / "trajectory_previews.jsonl"
TEMP_CONFIG_PATH = REPO_DIR / "tmp" / "colab_runtime" / f"{BASE_TRAIN_CONFIG.stem}_{GFLOWNET_OBJECTIVE}.yaml"
DEFAULT_PPO_FALLBACK_CHECKPOINT = "QizhiPei/biot5-plus-base-chebi20"
UPSTREAM_CHECKPOINT = REPO_DIR / "outputs" / "multi_molecule_sft_mini" / "checkpoints" / "best"
MINI_DATASET_DIR = REPO_DIR / "data" / "mini_post_training"
MINI_DATASET_CHECKS = {
    "train_multimol": MINI_DATASET_DIR / "grouped_splits" / "train_multimol.jsonl",
    "split_validation": MINI_DATASET_DIR / "grouped_splits" / "validation_multimol.jsonl",
    "split_test": MINI_DATASET_DIR / "grouped_splits" / "test_multimol.jsonl",
}

print({
    "base_train_config": str(BASE_TRAIN_CONFIG),
    "gflownet_objective": GFLOWNET_OBJECTIVE,
    "force_valid_masking": FORCE_VALID_MASKING,
    "selfies_dict_path": SELFIES_DICT_PATH,
    "rollout_temperature": ROLLOUT_TEMPERATURE,
    "rollout_top_p": ROLLOUT_TOP_P,
    "output_dir": str(OUTPUT_DIR),
})


In [ ]:
import os
import wandb

# Temporary Colab login bootstrap. Remove before committing or sharing this notebook.
WANDB_API_KEY = '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'
os.environ["WANDB_API_KEY"] = WANDB_API_KEY
wandb.login(key=WANDB_API_KEY, relogin=True)
print({"wandb_api_key_configured": True, "wandb_project_url": WANDB_PROJECT_URL})


In [ ]:
%cd {REPO_DIR}
bootstrap_command = [
    sys.executable,
    "scripts/init_colab.py",
    "--stage",
    "gflownet",
    "--repo-url",
    REPO_URL,
    "--repo-branch",
    REPO_BRANCH,
    "--repo-dir",
    str(REPO_DIR),
    "--dataset-mode",
    DATASET_MODE,
    "--train-dataset-file-id",
    TRAIN_DATASET_FILE_ID,
]
if CONFIG_OVERRIDE:
    bootstrap_command.extend(["--config", str(CONFIG_OVERRIDE)])

print("Bootstrapping:", " ".join(str(part) for part in bootstrap_command))

def run_and_stream(command):
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    if process.stdout is None:
        raise RuntimeError("Failed to capture command output.")

    try:
        for line in process.stdout:
            print(line, end="", flush=True)
    finally:
        process.stdout.close()

    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

run_and_stream(bootstrap_command)

import yaml

TEMP_CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
runtime_config = yaml.safe_load(BASE_TRAIN_CONFIG.read_text())
runtime_config.setdefault("gflownet", {})["objective"] = GFLOWNET_OBJECTIVE
runtime_config.setdefault("training", {})["output_dir"] = str(OUTPUT_DIR)

gflownet_payload = runtime_config.setdefault("gflownet", {})
rollout_payload = dict(gflownet_payload.get("rollout", {}))
if FORCE_VALID_MASKING:
    rollout_payload["constrained_decoding"] = True
    rollout_payload["selfies_dict_path"] = SELFIES_DICT_PATH
if ROLLOUT_TEMPERATURE is not None:
    rollout_payload["temperature"] = float(ROLLOUT_TEMPERATURE)
if ROLLOUT_TOP_P is not None:
    rollout_payload["top_p"] = float(ROLLOUT_TOP_P)
gflownet_payload["rollout"] = rollout_payload

TEMP_CONFIG_PATH.write_text(yaml.safe_dump(runtime_config, sort_keys=False), encoding="utf-8")
print({
    "runtime_config": str(TEMP_CONFIG_PATH),
    "gflownet_objective": GFLOWNET_OBJECTIVE,
    "force_valid_masking": FORCE_VALID_MASKING,
    "rollout_temperature": rollout_payload.get("temperature"),
    "rollout_top_p": rollout_payload.get("top_p"),
    "constrained_decoding": rollout_payload.get("constrained_decoding"),
    "selfies_dict_path": rollout_payload.get("selfies_dict_path"),
})

if CONFIG_OVERRIDE is None and not UPSTREAM_CHECKPOINT.exists():
    print(
        "Default upstream checkpoint is missing at "
        f"{UPSTREAM_CHECKPOINT}; GFlowNet will fall back to {DEFAULT_PPO_FALLBACK_CHECKPOINT}."
    )

training_command = [
    sys.executable,
    "scripts/train_multi_molecule_gflownet.py",
    "--config",
    str(TEMP_CONFIG_PATH),
]
print("Training:", " ".join(str(part) for part in training_command))
run_and_stream(training_command)


In [ ]:
import json
import shutil

summary_payload = json.loads(RUN_SUMMARY_PATH.read_text()) if RUN_SUMMARY_PATH.exists() else {}
best_checkpoint_dir = Path(summary_payload["best_checkpoint_dir"]) if summary_payload.get("best_checkpoint_dir") else BEST_CHECKPOINT_DIR
best_checkpoint_zip = Path(summary_payload["best_checkpoint_zip"]) if summary_payload.get("best_checkpoint_zip") else BEST_CHECKPOINT_ZIP
output_dir_zip = None
if OUTPUT_DIR.exists():
    output_dir_zip = Path(
        shutil.make_archive(
            str(OUTPUT_DIR),
            "zip",
            root_dir=OUTPUT_DIR.parent,
            base_dir=OUTPUT_DIR.name,
        )
    )

print({
    "gflownet_objective": GFLOWNET_OBJECTIVE,
    "output_dir": str(OUTPUT_DIR),
    "output_dir_exists": OUTPUT_DIR.exists(),
    "output_dir_zip": str(output_dir_zip) if output_dir_zip else str(OUTPUT_DIR_ZIP),
    "output_dir_zip_exists": output_dir_zip is not None and output_dir_zip.exists(),
    "checkpoint_dir": str(CHECKPOINTS_DIR),
    "checkpoint_dir_exists": CHECKPOINTS_DIR.exists(),
    "saved_iteration_checkpoints": sorted(path.name for path in CHECKPOINTS_DIR.glob("iteration-*")) if CHECKPOINTS_DIR.exists() else [],
    "best_checkpoint_dir": str(best_checkpoint_dir),
    "best_checkpoint_dir_exists": best_checkpoint_dir.exists(),
    "best_checkpoint_zip": str(best_checkpoint_zip),
    "best_checkpoint_zip_exists": best_checkpoint_zip.exists(),
    "run_summary": str(RUN_SUMMARY_PATH),
    "run_summary_exists": RUN_SUMMARY_PATH.exists(),
    "best_objective_loss": summary_payload.get("best_objective_loss"),
    "best_checkpoint_iteration": summary_payload.get("best_checkpoint_iteration"),
    "resolved_checkpoint_source": summary_payload.get("resolved_checkpoint_source"),
    "iteration_diagnostics_path": str(ITERATION_DIAGNOSTICS_PATH),
    "iteration_diagnostics_exists": ITERATION_DIAGNOSTICS_PATH.exists(),
    "trajectory_previews_path": str(TRAJECTORY_PREVIEWS_PATH),
    "trajectory_previews_exists": TRAJECTORY_PREVIEWS_PATH.exists(),
    "upstream_checkpoint": str(UPSTREAM_CHECKPOINT),
    "upstream_checkpoint_exists": UPSTREAM_CHECKPOINT.exists(),
    "mini_dataset_dir": str(MINI_DATASET_DIR),
    "mini_dataset_exists": MINI_DATASET_DIR.exists(),
    "mini_dataset_checks": {name: path.exists() for name, path in MINI_DATASET_CHECKS.items()},
})
